In [2]:
!pip install -q --no-cache-dir "vllm==0.19.1"
!pip -q install -U transformers huggingface_hub
!pip install -U "protobuf>=5.26.1,<6"

In [3]:
import argparse
import os
import sys
from pathlib import Path
import time

ROOT = "/content/drive/MyDrive/Colab Notebooks/kisti/system/project_to_company"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# 전처리 결과 & LLM 생성 최종 결과 파일 저장 시 파일명 중복 방지를 위한 함수
def next_result_path(base_path="./result/result.csv"):
    base = Path(base_path)
    base.parent.mkdir(parents=True, exist_ok=True)

    if not base.exists():
        return str(base)

    stem = base.stem
    suffix = base.suffix
    parent = base.parent

    idx = 2
    while True:
        candidate = parent / f"{stem}_{idx}{suffix}"
        if not candidate.exists():
            return str(candidate)
        idx += 1

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60

    if hours > 0:
        return f"{hours}시간 {minutes}분 {secs:.2f}초"

    if minutes > 0:
        return f"{minutes}분 {secs:.2f}초"

    return f"{secs:.2f}초"


def main():
    print("모듈 및 모델 라이브러리 import 중 . . .")
    from data_pipeline import (
        load_data,
        select_project_matches,
        preprocess,
        load_embedding_model,
    )
    from llm_pipeline import load_model, run_generation
    print("import 완료")

    print("data.csv 로드 중 . . .")
    df = load_data("./data/data.csv")
    print("data.csv 로드 완료")

    print("임베딩 모델 로딩 중 . . .")
    t_embed_model_start = time.perf_counter()
    embed_model, embed_tokenizer, embed_device = load_embedding_model("BAAI/bge-m3")
    t_embed_model_end = time.perf_counter()
    print(f"[완료] 임베딩 모델 로딩: {format_time(t_embed_model_end - t_embed_model_start)}")

    print("LLM 모델 로딩 중 . . .")
    t_llm_model_start = time.perf_counter()
    llm, llm_tokenizer = load_model(
        model_id="Qwen/Qwen3.5-35B-A3B-GPTQ-Int4", #"./models/Qwen3.5-35B-A3B-GPTQ-Int4",
        hf_token=os.environ.get("HF_TOKEN"),
        tensor_parallel_size=1,
    )
    t_llm_model_end = time.perf_counter()
    print(f"[완료] LLM 모델 로딩: {format_time(t_llm_model_end - t_llm_model_start)}")

    while True:
        project_name = input("과제명을 입력하세요. 종료하려면 end 입력: ").strip()

        if project_name.lower() == "end":
            print("종료합니다.")
            break

        try:
            top_n = int(input("가져올 개수를 입력하세요: ").strip())
        except:
            print("숫자를 입력해주세요.")
            continue

        try:
          print("데이터 전처리 시작")
          t_pre_start = time.perf_counter()

          tmp = select_project_matches(
              df=df,
              project_name=project_name,
              top_n=top_n,
          )

          tmp = preprocess(
              tmp,
              embed_model=embed_model,
              embed_tokenizer=embed_tokenizer,
              embed_device=embed_device,
          )

          tmp_path = next_result_path("./data/tmp.csv")
          tmp.to_csv(tmp_path, index=False, encoding="utf-8-sig")

          t_pre_end = time.perf_counter()
          elapsed = t_pre_end - t_pre_start
          print(f"[완료] 데이터 전처리 완료: {format_time(elapsed)}")

          print(f"데이터 전처리 수: {len(tmp)}건")
          print(f"전처리 파일 저장: {tmp_path}")

          out_path = next_result_path("./result/result.csv")

          print(f"LLM 생성 시작: {out_path}")
          t_llm_start = time.perf_counter()

          result = run_generation(
              tmp=tmp,
              model=llm,
              tokenizer=llm_tokenizer,
              out_path=out_path,
              overwrite=False,
          )

          t_llm_end = time.perf_counter()
          elapsed = t_llm_end - t_llm_start
          print(f"[완료] llm 생성 완료: {format_time(elapsed)}")
          print(f"결과 파일: {result['out_path']}")

        except ValueError as e:
            print(f"[오류] {e}")
            print("다른 과제명을 입력하거나 end로 종료해주세요.")
            continue

        except Exception as e:
            print(f"[오류] 처리 중 문제가 발생했습니다: {e}")
            print("다른 과제명을 입력하거나 end로 종료해주세요.")
            continue


main()

모듈 및 모델 라이브러리 import 중 . . .
import 완료
data.csv 로드 중 . . .
data.csv 로드 완료
임베딩 모델 로딩 중 . . .


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[완료] 임베딩 모델 로딩: 6.67초
LLM 모델 로딩 중 . . .
INFO 06-08 07:29:15 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-35B-A3B-GPTQ-Int4'}
WARNING 06-08 07:29:15 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-08 07:29:17 [model.py:549] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 06-08 07:29:17 [model.py:1678] Using max model len 262144
INFO 06-08 07:29:18 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 06-08 07:29:18 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[완료] llm 생성 완료: 6.38초
결과 파일: result/result.csv
과제명을 입력하세요. 종료하려면 end 입력: 미래산업환경대응홀로닉생산시스템개발
가져올 개수를 입력하세요: 1
데이터 전처리 시작
[완료] 데이터 전처리 완료: 0.14초
데이터 전처리 수: 1건
전처리 파일 저장: data/tmp_2.csv
LLM 생성 시작: result/result_2.csv


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[완료] llm 생성 완료: 5.48초
결과 파일: result/result_2.csv
과제명을 입력하세요. 종료하려면 end 입력: end
종료합니다.
